# ⚡ LogShield — Notebook 02: PySpark Medallion Lakehouse ETL

This notebook demonstrates the Apache PySpark processing pipeline: building a Spark session, ingesting raw access logs into Bronze Parquet, normalizing into Silver Parquet, and executing Gold aggregations.

In [ ]:
from src.logshield.spark.session import get_spark_session
from pyspark.sql import functions as F

spark = get_spark_session(app_name="LogShield-Notebook-ETL")
print("Spark Version:", spark.version)
print("Master URL:", spark.sparkContext.master)

## 1. Inspect Silver Lakehouse Layer

In [ ]:
silver_df = spark.read.parquet("data/silver")
print(f"Total Silver Records: {silver_df.count():,}")
silver_df.printSchema()
silver_df.show(5, truncate=False)

## 2. Execute PySpark Aggregations (HTTP Status Breakdown)

In [ ]:
status_counts = silver_df.groupBy("status").count().orderBy(F.desc("count"))
status_counts.show()

## 3. Hourly 404 Error Distribution & Peak Analysis

In [ ]:
errors_404 = silver_df.filter(F.col("status") == 404)
hourly_404 = errors_404.groupBy("hour").count().orderBy("hour")
hourly_404.show(24)